# Train-Test Split and Class Balancing

This notebook prepares the feature-engineered dataset for machine learning by:

- Loading the engineered dataset
- Separating features and target
- Performing an 80/20 stratified train-test split
- Handling class imbalance using SMOTE
- Saving the prepared datasets

## Import Libraries

In [178]:
import pandas as pd

from sklearn.model_selection import train_test_split

from imblearn.over_sampling import SMOTE

## Load Dataset

In [179]:
df = pd.read_csv("../data/processed/creditcard_feature_engineered.csv")

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V24,V25,V26,V27,V28,Amount,Class,Large_Transaction,Log_Amount,Amount_Level
0,-1.996823,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.066928,0.128539,-0.189115,0.133558,-0.021053,0.244200,0,0,5.014760,3
1,-1.996823,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.342584,0,0,1.305626,0
2,-1.996802,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,1.158900,0,1,5.939276,3
3,-1.996802,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-1.175575,0.647376,-0.221929,0.062723,0.061458,0.139886,0,0,4.824306,3
4,-1.996781,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.141267,-0.206010,0.502292,0.219422,0.215153,-0.073813,0,0,4.262539,2


## Dataset Information

In [180]:
print("Shape:", df.shape)

df.info()

Shape: (283726, 34)
<class 'pandas.DataFrame'>
RangeIndex: 283726 entries, 0 to 283725
Data columns (total 34 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Time               283726 non-null  float64
 1   V1                 283726 non-null  float64
 2   V2                 283726 non-null  float64
 3   V3                 283726 non-null  float64
 4   V4                 283726 non-null  float64
 5   V5                 283726 non-null  float64
 6   V6                 283726 non-null  float64
 7   V7                 283726 non-null  float64
 8   V8                 283726 non-null  float64
 9   V9                 283726 non-null  float64
 10  V10                283726 non-null  float64
 11  V11                283726 non-null  float64
 12  V12                283726 non-null  float64
 13  V13                283726 non-null  float64
 14  V14                283726 non-null  float64
 15  V15                283726 non-null  float6

## Label Encoding

In [181]:
from sklearn.preprocessing import LabelEncoder

In [182]:
encoder = LabelEncoder()
df["Amount_Level"] = encoder.fit_transform(df["Amount_Level"])

In [183]:
df["Amount_Level"].value_counts()

Amount_Level
1    71076
0    70938
3    70929
2    70783
Name: count, dtype: int64

In [184]:
df["Amount_Level"].unique()

array([3, 0, 2, 1])

## Separate Features and Target

In [185]:
X = df.drop("Class", axis=1)
y = df["Class"]

### Verify

In [186]:
print("Features shape :", X.shape)
print("Target shape :", y.shape)

Features shape : (283726, 33)
Target shape : (283726,)


## Check Class Distributio

In [187]:
y.value_counts()

Class
0    283253
1       473
Name: count, dtype: int64

In [188]:
y.value_counts(normalize=True) * 100

Class
0    99.83329
1     0.16671
Name: proportion, dtype: float64

The dataset is highly imbalanced with fraudulent transactions representing a very small percentage of all transactions.

## Train-Test Split

In [189]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Verify Shapes

In [190]:
print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Testing Labels :", y_test.shape)

Training Features : (226980, 33)
Testing Features : (56746, 33)
Training Labels : (226980,)
Testing Labels : (56746,)


## Verify Class Distribution After Split

In [191]:
y_train.value_counts()

Class
0    226602
1       378
Name: count, dtype: int64

In [192]:
y_test.value_counts()

Class
0    56651
1       95
Name: count, dtype: int64

In [193]:
print(y_train.value_counts(normalize=True))

print(y_test.value_counts(normalize=True))

Class
0    0.998335
1    0.001665
Name: proportion, dtype: float64
Class
0    0.998326
1    0.001674
Name: proportion, dtype: float64


Stratified sampling preserved the fraud/non-fraud ratio in both training and testing datasets.

In [194]:
X_train.select_dtypes(include=["object", "string"]).columns

Index([], dtype='str')

## Apply SMOTE

In [195]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

### Verify

In [196]:
print(X_train_smote.shape)

print(y_train_smote.value_counts())

(453204, 33)
Class
0    226602
1    226602
Name: count, dtype: int64


## Compare Before vs After SMOTE

In [197]:
print("Original")

print(y_train.value_counts())

print()

print("After SMOTE")

print(y_train_smote.value_counts())

Original
Class
0    226602
1       378
Name: count, dtype: int64

After SMOTE
Class
0    226602
1    226602
Name: count, dtype: int64


## Save Prepared Data

In [198]:
X_train_smote.to_csv("../data/processed/X_train.csv", index=False)

X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train_smote.to_csv("../data/processed/y_train.csv", index=False)

y_test.to_csv("../data/processed/y_test.csv", index=False)

###  Verify

In [199]:
import os

os.listdir("../data/processed")

['creditcard_feature_engineered.csv',
 '.gitkeep',
 'creditcard_cleaned.csv',
 'X_train.csv',
 'y_train.csv',
 'y_test.csv',
 'X_test.csv']

In [200]:
print("Training Data")

print(X_train_smote.shape)

print(y_train_smote.shape)

print()

print("Testing Data")

print(X_test.shape)

print(y_test.shape)

Training Data
(453204, 33)
(453204,)

Testing Data
(56746, 33)
(56746,)


## Final Observation

- Loaded feature-engineered dataset.
- Verified dataset structure and class distribution.
- Encoded the Amount_Level categorical feature using LabelEncoder.
- Split the dataset into 80% training and 20% testing using stratified sampling.
- Applied SMOTE only on the training dataset to handle class imbalance.
- Successfully created balanced training data while keeping the test set untouched.
- Saved all prepared datasets for the model-building phase.